# Analisis EDA para el conjunto de datos 

antes de empezar instalamos la libreria pyarrow la cual nos permita convertir los datos a formato parquet. el cual es un formato de archivo columnar que facilita el tratamiento y manipulacion de grandes volumenes de informacon

In [1]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


DuckDB, una librería que lee CSV enormes sin cargarlos completos en memoria y los convierte a parquet, uniendo los cinco años en un solo paso. También detecta los tipos de datos por ti

In [7]:
%pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 28.1 MB/s  0:00:00m0:00:010:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd

carpeta_datos = Path("Flight_records")                      # la carpeta donde está el notebook
carpeta_parquet = Path("parquet_por_anio")     # aquí guardaremos un parquet por año
carpeta_parquet.mkdir(exist_ok=True)

archivos_csv = sorted(carpeta_datos.glob("Combined_Flights_*.csv"))

for archivo in archivos_csv:
    print(archivo.name)

Convertir y unir los 5 años de registros de vuelos 

In [8]:
import duckdb
from pathlib import Path

archivo_final = Path("Combined_Flights_2018_2022.parquet")

duckdb.sql(f"""
    COPY (
        SELECT *
        FROM read_csv('Combined_Flights_20??.csv', union_by_name = true, sample_size = -1)
    )
    TO '{archivo_final}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Listo:", archivo_final, f"({archivo_final.stat().st_size / 1e9:.1f} GB)")

Listo: Combined_Flights_2018_2022.parquet (0.8 GB)


Conteo de vuelos por año

In [9]:
import pyarrow.parquet as pq

esquema = pq.read_schema(archivo_final)
print("Columnas:", len(esquema.names))

duckdb.sql(f"""
    SELECT Year, COUNT(*) AS vuelos
    FROM '{archivo_final}'
    GROUP BY Year
    ORDER BY Year
""").df()

Columnas: 61


,Year,vuelos
0,2018,5689512
1,2019,8091684
2,2020,5022397
3,2021,6311871
4,2022,4078318


Vista parcial del dataframe

In [10]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 70)   # mostrar todas las columnas
pd.set_option("display.max_rows", 100)     # mostrar hasta 100 filas

ruta = "Combined_Flights_2018_2022.parquet"

duckdb.sql(f"SELECT * FROM '{ruta}' LIMIT 5").df()

,FlightDate,Airline,Origin,Dest,Cancelled,Diverted,CRSDepTime,DepTime,DepDelayMinutes,DepDelay,ArrTime,ArrDelayMinutes,AirTime,CRSElapsedTime,ActualElapsedTime,Distance,Year,Quarter,Month,DayofMonth,DayOfWeek,Marketing_Airline_Network,Operated_or_Branded_Code_Share_Partners,DOT_ID_Marketing_Airline,IATA_Code_Marketing_Airline,Flight_Number_Marketing_Airline,Operating_Airline,DOT_ID_Operating_Airline,IATA_Code_Operating_Airline,Tail_Number,Flight_Number_Operating_Airline,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,DestCityName,DestState,DestStateFips,DestStateName,DestWac,DepDel15,DepartureDelayGroups,DepTimeBlk,TaxiOut,WheelsOff,WheelsOn,TaxiIn,CRSArrTime,ArrDelay,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,DistanceGroup,DivAirportLandings
0,2018-01-23,Endeavor Air Inc.,ABY,ATL,False,False,1202,1157.0,0.0,-5.0,1256.0,0.0,38.0,62.0,59.0,145.0,2018,1,1,23,2,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N8928A,3298,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,0.0,-1.0,1200-1259,14.0,1211.0,1249.0,7.0,1304,-8.0,0.0,-1.0,1300-1359,1,0.0
1,2018-01-24,Endeavor Air Inc.,ABY,ATL,False,False,1202,1157.0,0.0,-5.0,1258.0,0.0,36.0,62.0,61.0,145.0,2018,1,1,24,3,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N800AY,3298,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,0.0,-1.0,1200-1259,13.0,1210.0,1246.0,12.0,1304,-6.0,0.0,-1.0,1300-1359,1,0.0
2,2018-01-25,Endeavor Air Inc.,ABY,ATL,False,False,1202,1153.0,0.0,-9.0,1302.0,0.0,40.0,62.0,69.0,145.0,2018,1,1,25,4,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N8836A,3298,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,0.0,-1.0,1200-1259,18.0,1211.0,1251.0,11.0,1304,-2.0,0.0,-1.0,1300-1359,1,0.0
3,2018-01-26,Endeavor Air Inc.,ABY,ATL,False,False,1202,1150.0,0.0,-12.0,1253.0,0.0,35.0,62.0,63.0,145.0,2018,1,1,26,5,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N800AY,3298,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,0.0,-1.0,1200-1259,17.0,1207.0,1242.0,11.0,1304,-11.0,0.0,-1.0,1300-1359,1,0.0
4,2018-01-27,Endeavor Air Inc.,ABY,ATL,False,False,1400,1355.0,0.0,-5.0,1459.0,0.0,36.0,60.0,64.0,145.0,2018,1,1,27,6,DL,DL_CODESHARE,19790,DL,3298,9E,20363,9E,N8839E,3298,10146,1014602,30146,"Albany, GA",GA,13,Georgia,34,10397,1039707,30397,"Atlanta, GA",GA,13,Georgia,34,0.0,-1.0,1400-1459,17.0,1412.0,1448.0,11.0,1500,-1.0,0.0,-1.0,1500-1559,1,0.0


Tipo de los datos

In [11]:
total = duckdb.sql(f"SELECT COUNT(*) FROM '{ruta}'").fetchone()[0]
print(f"Total de vuelos: {total:,}")

duckdb.sql(f"DESCRIBE SELECT * FROM '{ruta}'").df()[["column_name", "column_type"]]

Total de vuelos: 29,193,782


,column_name,column_type
0,FlightDate,DATE
1,Airline,VARCHAR
2,Origin,VARCHAR
3,Dest,VARCHAR
4,Cancelled,BOOLEAN
5,Diverted,BOOLEAN
6,CRSDepTime,BIGINT
7,DepTime,DOUBLE
8,DepDelayMinutes,DOUBLE
9,DepDelay,DOUBLE


Resumen de las columnas con nulos

In [12]:
resumen = duckdb.sql(f"SUMMARIZE SELECT * FROM '{ruta}'").df()

resumen[["column_name", "column_type", "null_percentage", "approx_unique", "min", "max"]]

,column_name,column_type,null_percentage,approx_unique,min,max
0,FlightDate,DATE,0.00,1914,2018-01-01,2022-07-31
1,Airline,VARCHAR,0.00,30,Air Wisconsin Airlines Corp,Virgin America
2,Origin,VARCHAR,0.00,432,ABE,YUM
3,Dest,VARCHAR,0.00,432,ABE,YUM
4,Cancelled,BOOLEAN,0.00,2,false,true
5,Diverted,BOOLEAN,0.00,2,false,true
6,CRSDepTime,BIGINT,0.00,1379,1,2359
7,DepTime,DOUBLE,2.61,1700,1.0,2400.0
8,DepDelayMinutes,DOUBLE,2.61,2156,0.0,7223.0
9,DepDelay,DOUBLE,2.61,2387,-1280.0,7223.0


Colummnas con mas nulos

In [13]:
resumen[resumen["null_percentage"] > 0] \
    .sort_values("null_percentage", ascending=False) \
    [["column_name", "null_percentage"]]

,column_name,null_percentage
12,AirTime,2.92
57,ArrivalDelayGroups,2.90
56,ArrDel15,2.90
11,ArrDelayMinutes,2.90
14,ActualElapsedTime,2.90
55,ArrDelay,2.90
53,TaxiIn,2.72
52,WheelsOn,2.72
10,ArrTime,2.69
51,WheelsOff,2.67


Vuelos, cancelaciones y desvios por año

In [14]:
duckdb.sql(f"""
    SELECT
        Year,
        COUNT(*)                                           AS vuelos,
        ROUND(100 * AVG(CAST(Cancelled AS INT)), 2)        AS pct_cancelados,
        ROUND(100 * AVG(CAST(Diverted  AS INT)), 2)        AS pct_desviados
    FROM '{ruta}'
    GROUP BY Year
    ORDER BY Year
""").df()

,Year,vuelos,pct_cancelados,pct_desviados
0,2018,5689512,1.55,0.25
1,2019,8091684,1.90,0.26
2,2020,5022397,5.99,0.17
3,2021,6311871,1.76,0.24
4,2022,4078318,3.02,0.25


Que columnas hay y que meses cubre cada uno?

In [15]:
columnas = duckdb.sql(f"DESCRIBE SELECT * FROM '{ruta}'").df()["column_name"].tolist()

for c in ["CarrierDelay", "WeatherDelay", "NASDelay", "LateAircraftDelay", "Div1Airport"]:
    print(f"{c:<20} {'está' if c in columnas else 'NO está'}")

cobertura = duckdb.sql(f"""
    SELECT Year, Month, COUNT(*) AS vuelos
    FROM '{ruta}'
    GROUP BY Year, Month
""").df()

cobertura.pivot(index="Year", columns="Month", values="vuelos")

CarrierDelay         NO está
WeatherDelay         NO está
NASDelay             NO está
LateAircraftDelay    NO está
Div1Airport          NO está


Month,1,2,3,4,5,6,7,8,9,10,11,12
Year,,,,,,,,,,,,
2018,457688.0,268105.0,346132.0,482983.0,350044.0,317713.0,367221.0,503612.0,637614.0,670090.0,638985.0,649325.0
2019,638649.0,582966.0,689221.0,668259.0,694311.0,694469.0,717684.0,717456.0,660712.0,692944.0,655072.0,679941.0
2020,660556.0,623103.0,701274.0,331238.0,192412.0,237264.0,370859.0,398470.0,345294.0,374538.0,389587.0,397802.0
2021,379384.0,350170.0,467126.0,473936.0,520059.0,573779.0,615703.0,611494.0,567916.0,595373.0,576693.0,580238.0
2022,563737.0,519952.0,590542.0,580290.0,602950.0,602057.0,618790.0,NaN,NaN,NaN,NaN,NaN


Averiguar que los nulos son vuelos desviados o cancelados

In [16]:
duckdb.sql(f"""
    SELECT
        Cancelled,
        Diverted,
        COUNT(*) AS vuelos,
        ROUND(100 * AVG(CAST(DepDelay IS NULL AS INT)), 2) AS pct_sin_DepDelay,
        ROUND(100 * AVG(CAST(ArrDelay IS NULL AS INT)), 2) AS pct_sin_ArrDelay
    FROM '{ruta}'
    GROUP BY Cancelled, Diverted
    ORDER BY Cancelled, Diverted
""").df()

,Cancelled,Diverted,vuelos,pct_sin_DepDelay,pct_sin_ArrDelay
0,False,False,28348168,0.00,0.0
1,False,True,68347,0.01,100.0
2,True,False,777265,98.03,100.0
3,True,True,2,100.00,100.0


Como resultado de la consulta observamos que los valores nulos presentes en el dataframe corresponden a vuelos cancelados o desviados. 777.265 de los nulos corresponden a vuelos cancelados y 68.347 fueron desviados. 

VUELOS QUE SE COMPLETARON

In [17]:
duckdb.sql(f"""
    CREATE OR REPLACE VIEW vuelos_operados AS
    SELECT *
    FROM '{ruta}'
    WHERE Cancelled = FALSE AND Diverted = FALSE
""")

duckdb.sql("""
    SELECT
        COUNT(*) AS vuelos_operados,
        SUM(CAST(ArrDelay IS NULL AS INT)) AS sin_ArrDelay
    FROM vuelos_operados
""").df()

,vuelos_operados,sin_ArrDelay
0,28348168,569.0


RETRASOS POR AÑO

In [18]:
duckdb.sql("""
    SELECT
        Year,
        COUNT(*)                          AS vuelos,
        ROUND(AVG(ArrDelay), 1)           AS retraso_medio_min,
        ROUND(100 * AVG(ArrDel15), 1)     AS pct_retrasados_15min
    FROM vuelos_operados
    GROUP BY Year
    ORDER BY Year
""").df()

,Year,vuelos,retraso_medio_min,pct_retrasados_15min
0,2018,5587186,5.4,19.5
1,2019,7917264,5.7,19.3
2,2020,4712931,-4.9,10.0
3,2021,6185871,3.3,17.3
4,2022,3944916,7.5,21.6
